# Time-Distance Map Example: 2011 August 17

This notebook demonstrates the time-distance analysis used for one polar crown filament eruption observed by SDO/AIA 304 A. The workflow is:

1. Load an AIA image sequence.
2. Manually define a slit from the visible limb outward along the eruption direction.
3. Build a time-distance map by sampling the intensity along the slit.
4. Manually select the filament-front height-time points.
5. Fit the selected points with an exponential-plus-linear model to estimate the fast-rise onset.
6. Fit the last five selected points with a straight line to estimate the late-phase projected velocity.

Raw FITS files are not included in this repository. Update `DATA_GLOB` to the local path of your AIA 304 A data before running the notebook.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sunpy.map
import astropy.units as u

from pathlib import Path
from astropy.time import Time
from astropy.coordinates import SkyCoord
from scipy.optimize import curve_fit
from skimage.measure import profile_line

# Local AIA 304 A data path. Edit this path for a different event or machine.
DATA_GLOB = "/Volumes/Ying_Disk/research/raw_data/pcf/2011.08.17_03_28_00_TAI/*.image_lev1.fits"

# Saved manual selections. Set the FORCE_* switches to True to overwrite them.
SLIT_POINTS_FILE = Path("selected_slit_points.csv")
FIT_POINTS_FILE = Path("selected_fit_points_reviewed.csv")
FORCE_RESELECT_SLIT = False
FORCE_RESELECT_FIT = False

# Analysis settings.
ARCSEC_TO_KM = 725.0


## Load the AIA Image Sequence


In [ ]:
sequence_304 = sunpy.map.Map(DATA_GLOB, sequence=True)
example_map = sequence_304[0]

print(f"Number of frames: {len(sequence_304)}")
print(f"First frame: {sequence_304[0].date.isot}")
print(f"Last frame:  {sequence_304[-1].date.isot}")


## Select or Load the Slit

The slit is defined by two manually selected image-pixel positions. The first point should be placed near the visible limb at the eruption site, and the second point should extend outward along the main eruption direction.


In [ ]:
%matplotlib qt

if SLIT_POINTS_FILE.exists() and not FORCE_RESELECT_SLIT:
    saved_slit = np.loadtxt(SLIT_POINTS_FILE, delimiter=",", skiprows=1)
    saved_slit = np.atleast_2d(saved_slit)
    pts = [(saved_slit[0, 0], saved_slit[0, 1]), (saved_slit[1, 0], saved_slit[1, 1])]
    print("Loaded saved slit points:", pts)
else:
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection=example_map)
    example_map.plot(axes=ax, clip_interval=(1, 99.5) * u.percent)
    ax.set_title("Select two slit endpoints, then press Enter")

    pts = plt.ginput(n=2, timeout=0)
    plt.close(fig)

    if len(pts) != 2:
        raise RuntimeError("Two slit endpoints are required.")

    np.savetxt(
        SLIT_POINTS_FILE,
        np.asarray(pts, dtype=float),
        delimiter=",",
        header="x_pixel,y_pixel",
        comments="",
    )
    print("Saved slit points:", pts)

(x0, y0), (x1, y1) = pts
p0 = (y0, x0)  # profile_line uses (row, column)
p1 = (y1, x1)

w0 = example_map.pixel_to_world(x0 * u.pixel, y0 * u.pixel)
w1 = example_map.pixel_to_world(x1 * u.pixel, y1 * u.pixel)
slit_coords = SkyCoord([w0, w1])

print("Slit endpoints in image pixels:", pts)
print("Slit endpoints in helioprojective coordinates:")
print(slit_coords)


In [ ]:
%matplotlib inline

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection=example_map)
example_map.plot(axes=ax, clip_interval=(1, 99.5) * u.percent)
ax.plot_coord(slit_coords, color="red", linewidth=2)
ax.set_title("AIA 304 A Slit Position")
fig.savefig("aia_slit_example.png", dpi=300, bbox_inches="tight")
fig.savefig("aia_slit_example.pdf", bbox_inches="tight")
plt.show()


## Build the Time-Distance Map

For each AIA frame, the intensity profile is sampled along the same slit. The distance along the slit is measured in angular units on the plane of the sky and converted to a projected physical distance using 1 arcsec = 725 km.


In [ ]:
profiles = []
frame_times = []

for aia_map in sequence_304:
    profile = profile_line(
        aia_map.data,
        p0,
        p1,
        mode="constant",
        cval=np.nan,
        order=1,
        linewidth=1,
    )
    profiles.append(profile)
    frame_times.append(aia_map.date)

profiles = np.asarray(profiles)
frame_times = Time(frame_times)
t_sec = (frame_times - frame_times[0]).to_value("s")
t_hr = t_sec / 3600.0

slit_length_arcsec = w0.separation(w1).to_value(u.arcsec)
height_arcsec = np.linspace(0.0, slit_length_arcsec, profiles.shape[1])
height_km = height_arcsec * ARCSEC_TO_KM
height_Mm = height_km / 1000.0

print(f"Time range: {t_hr[0]:.2f}--{t_hr[-1]:.2f} hr")
print(f"Slit length: {slit_length_arcsec:.1f} arcsec = {height_Mm[-1]:.1f} Mm")


In [ ]:
%matplotlib inline

vmin, vmax = np.nanpercentile(profiles, [5, 99])

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(
    profiles.T,
    origin="lower",
    aspect="auto",
    extent=[t_hr[0], t_hr[-1], height_Mm[0], height_Mm[-1]],
    cmap="gray",
    vmin=vmin,
    vmax=vmax,
)
ax.set_xlabel("Time since start [hours]")
ax.set_ylabel("Projected distance along slit [Mm]")
ax.set_title("Time-Distance Map")
fig.colorbar(im, ax=ax, label="AIA 304 A intensity [DN]")
fig.savefig("time_distance_map.png", dpi=300, bbox_inches="tight")
fig.savefig("time_distance_map.pdf", bbox_inches="tight")
plt.show()


## Select or Load Height-Time Points

Select the visible filament-front trajectory on the time-distance map. The selected points are saved so the fitting can be repeated without manually selecting them again.


In [ ]:
%matplotlib qt

if FIT_POINTS_FILE.exists() and not FORCE_RESELECT_FIT:
    selected_points = np.loadtxt(FIT_POINTS_FILE, delimiter=",", skiprows=1)
    selected_points = np.atleast_2d(selected_points)
    print(f"Loaded {len(selected_points)} saved height-time points.")
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(
        profiles.T,
        origin="lower",
        aspect="auto",
        extent=[t_hr[0], t_hr[-1], height_Mm[0], height_Mm[-1]],
        cmap="gray",
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_xlabel("Time since start [hours]")
    ax.set_ylabel("Projected distance along slit [Mm]")
    ax.set_title("Select filament-front points, then press Enter")

    clicked_points = plt.ginput(n=-1, timeout=0)
    plt.close(fig)

    if len(clicked_points) < 3:
        raise RuntimeError("At least three height-time points are required for fitting.")

    selected_points = np.asarray(clicked_points, dtype=float)
    np.savetxt(
        FIT_POINTS_FILE,
        selected_points,
        delimiter=",",
        header="time_hr,height_Mm",
        comments="",
    )
    print(f"Saved {len(selected_points)} height-time points.")

selected_points = selected_points[np.argsort(selected_points[:, 0])]
t_fit_sec = selected_points[:, 0] * 3600.0
h_fit_km = selected_points[:, 1] * 1000.0


In [ ]:
%matplotlib inline

fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(
    profiles.T,
    origin="lower",
    aspect="auto",
    extent=[t_hr[0], t_hr[-1], height_Mm[0], height_Mm[-1]],
    cmap="gray",
    vmin=vmin,
    vmax=vmax,
)
ax.scatter(t_fit_sec / 3600.0, h_fit_km / 1000.0, s=25, color="tab:red", label="Selected points")
ax.set_xlabel("Time since start [hours]")
ax.set_ylabel("Projected distance along slit [Mm]")
ax.legend()
fig.savefig("selected_points_on_tmap.png", dpi=300, bbox_inches="tight")
fig.savefig("selected_points_on_tmap.pdf", bbox_inches="tight")
plt.show()


## Exponential-Plus-Linear Fit

The height-time profile is fitted with

\[
h(t) = c_0\exp\left(rac{t-t_0}{	au}ight) + c_1(t-t_0) + c_2 .
\]

Here `t0` is fixed to the time of the first selected point to reduce parameter degeneracy. The fast-rise onset is defined as the time when the exponential velocity equals the linear velocity:

\[
t_{m onset} = t_0 + 	au \ln\left(rac{c_1	au}{c_0}ight).
\]


In [ ]:
def h_model_fixed_t0(t, c0, tau, c1, c2):
    return c0 * np.exp((t - t0_param) / tau) + c1 * (t - t0_param) + c2


def velocity_model(t, c0, tau, c1):
    return (c0 / tau) * np.exp((t - t0_param) / tau) + c1


t0_param = t_fit_sec[0]
c0_guess = max((h_fit_km[-1] - h_fit_km[0]) * 0.1, 1.0)
tau_guess = max((t_fit_sec[-1] - t_fit_sec[0]) / 3.0, 1.0)
c1_guess = max((h_fit_km[-1] - h_fit_km[0]) / (t_fit_sec[-1] - t_fit_sec[0]), 0.01)
c2_guess = h_fit_km[0]

p0_fit = [c0_guess, tau_guess, c1_guess, c2_guess]
bounds = ([0.0, 1.0, 0.0, -np.inf], [np.inf, np.inf, np.inf, np.inf])

popt, pcov = curve_fit(
    h_model_fixed_t0,
    t_fit_sec,
    h_fit_km,
    p0=p0_fit,
    bounds=bounds,
    maxfev=20000,
)

c0, tau, c1, c2 = popt
perr = np.sqrt(np.diag(pcov))

if c0 > 0 and tau > 0 and c1 > 0:
    t_onset = t0_param + tau * np.log(c1 * tau / c0)
else:
    t_onset = np.nan

if np.isfinite(t_onset):
    h_onset = h_model_fixed_t0(t_onset, *popt)
    v_onset = velocity_model(t_onset, c0, tau, c1)
else:
    h_onset = np.nan
    v_onset = np.nan

print("Fit parameters:")
print(f"c0  = {c0:.3g} +/- {perr[0]:.3g} km")
print(f"tau = {tau:.3g} +/- {perr[1]:.3g} s")
print(f"c1  = {c1:.3g} +/- {perr[2]:.3g} km/s")
print(f"c2  = {c2:.3g} +/- {perr[3]:.3g} km")
print(f"t_onset = {t_onset / 3600.0:.3f} hr")
print(f"H_onset = {h_onset / 1000.0:.2f} Mm")
print(f"v_onset = {v_onset:.2f} km/s")


In [ ]:
%matplotlib inline

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "legend.fontsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

t_dense = np.linspace(t_fit_sec[0], t_fit_sec[-1], 500)
h_dense_Mm = h_model_fixed_t0(t_dense, *popt) / 1000.0

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(t_fit_sec / 3600.0, h_fit_km / 1000.0, s=28, color="tab:blue", label="Selected points")
ax.plot(t_dense / 3600.0, h_dense_Mm, color="tab:red", linewidth=2, label="Fit")

if np.isfinite(t_onset):
    h_onset_Mm = h_onset / 1000.0
    ax.axvline(t_onset / 3600.0, color="black", linestyle="--", linewidth=1.4, label=r"$t_{\rm onset}$")
    ax.axhline(h_onset_Mm, color="black", linestyle=":", linewidth=1.4)

    dt_tangent = 10.0 * 60.0
    t_tangent = np.array([t_onset - dt_tangent, t_onset + dt_tangent])
    h_tangent = h_onset + v_onset * (t_tangent - t_onset)
    ax.plot(
        t_tangent / 3600.0,
        h_tangent / 1000.0,
        color="0.45",
        linewidth=1.5,
        label=r"Velocity at $t_{\rm onset}$",
    )

    ax.text(
        0.04,
        0.96,
        f"H_onset = {h_onset_Mm:.1f} Mm\n"
        f"v_onset = {v_onset:.1f} km/s\n"
        f"t_onset = {t_onset / 3600.0:.2f} hr",
        transform=ax.transAxes,
        va="top",
        ha="left",
        bbox=dict(facecolor="white", edgecolor="0.85", alpha=0.9),
    )

ax.set_xlabel("Time since start [hours]")
ax.set_ylabel("Projected distance along slit [Mm]")
ax.legend(loc="best")
fig.savefig("fitting_image_uniform.png", dpi=300, bbox_inches="tight")
fig.savefig("fitting_image_uniform.pdf", bbox_inches="tight")
plt.show()


## Late-Phase Linear Velocity

The late-phase projected velocity is estimated from a linear fit to the last five selected height-time points.


In [ ]:
%matplotlib inline

n_late = min(5, len(t_fit_sec))
if n_late < 2:
    raise RuntimeError("At least two selected points are required for the late-phase linear fit.")

t_late_sec = t_fit_sec[-n_late:]
h_late_km = h_fit_km[-n_late:]

linear_coeff = np.polyfit(t_late_sec, h_late_km, 1)
v_late = linear_coeff[0]
h_intercept = linear_coeff[1]

h_late_fit_km = np.polyval(linear_coeff, t_late_sec)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(t_fit_sec / 3600.0, h_fit_km / 1000.0, s=24, color="0.65", label="All selected points")
ax.scatter(t_late_sec / 3600.0, h_late_km / 1000.0, s=35, color="tab:green", label="Last five points")
ax.plot(
    t_late_sec / 3600.0,
    h_late_fit_km / 1000.0,
    color="tab:green",
    linewidth=2,
    label="Linear fit",
)
ax.text(
    0.04,
    0.96,
    f"v_late = {v_late:.1f} km/s",
    transform=ax.transAxes,
    va="top",
    ha="left",
    bbox=dict(facecolor="white", edgecolor="0.85", alpha=0.9),
)
ax.set_xlabel("Time since start [hours]")
ax.set_ylabel("Projected distance along slit [Mm]")
ax.legend(loc="best")
fig.savefig("late5_linear_fit_uniform.png", dpi=300, bbox_inches="tight")
fig.savefig("late5_linear_fit_uniform.pdf", bbox_inches="tight")
plt.show()

print(f"Late-phase velocity from the last {n_late} points: {v_late:.2f} km/s")
